# Failure Mode 6: Hallucinated Completion

> Before starting, read the [project README](../../README.md) for details on failure modes, scorers, and expectations.

The agent's final response is not grounded in evidence from its trace — it either contradicts what tools actually returned, or claims to have performed actions that don't appear in the trace.

### Introducing `@scorer` — custom evaluation logic

The [Graceful Refusal](../05_graceful_refusal/05_graceful_refusal.ipynb) notebook introduced `make_judge()` for building custom LLM judges. This notebook introduces the second pattern: `@scorer` — writing custom evaluation logic in Python.

Here we use `@scorer` as a wrapper around MLflow's built-in `is_grounded()` judge, extracting tool outputs from the trace and passing them as context. A later notebook shows `@scorer` used for pure deterministic logic with no LLM involved.

| Scorer | Source | Needs expectations? | What it checks |
|---|---|---|---|
| `grounded_in_tools` | Custom `@scorer` wrapping MLflow's `is_grounded()` | No | Whether the agent's response is supported by tool outputs in the trace |

For a detailed explanation of this failure mode and how the scorer works, see [hallucinated_completion.md](hallucinated_completion.md).

### Prerequisites and setup

Start a local MLflow server before running this notebook:

```bash
mlflow server --host 127.0.0.1 --port 5000
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import mlflow
from mlflow.entities import SpanType
from tools import TRAVEL_AGENT_TOOLS
from utils import print_eval_results

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("agentic-evaluation")
mlflow.tracing.disable_notebook_display()

EXPERIMENT = mlflow.get_experiment_by_name("agentic-evaluation")

# Clean up old traces for this failure mode
client = mlflow.MlflowClient()
old_traces = mlflow.search_traces(
    experiment_ids=[EXPERIMENT.experiment_id],
    filter_string="tags.failure_mode = 'hallucinated_completion'",
    return_type="list",
)
if old_traces:
    client.delete_traces(
        experiment_id=EXPERIMENT.experiment_id,
        trace_ids=[t.info.trace_id for t in old_traces],
    )
    print(f"Cleaned up {len(old_traces)} old traces.")

### Create traces

We create synthetic traces for a travel booking agent handling "Book me a flight from NYC to London on July 20." Four scenarios:

- **Ungrounded response (fail):** Tool returns price $450, agent says $299
- **Fabricated action — tool failed (fail):** Tool returns failure, agent claims booking succeeded
- **Fabricated action — no tool called (fail):** Agent claims booking without calling any tool
- **Grounded response (pass):** Agent accurately reflects the tool output

In [ ]:
# --- Failing trace: agent contradicts tool output (wrong price) ---
@mlflow.trace(name="travel_agent", span_type=SpanType.AGENT)
def hallucination_ungrounded(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, TRAVEL_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "hallucinated_completion", "expected_result": "fail"}
    )

    with mlflow.start_span(name="search_and_book", span_type=SpanType.TOOL) as span:
        span.set_inputs({"from_city": "NYC", "to_city": "London", "date": "2026-07-20"})
        span.set_outputs({
            "booking_id": "BK-456",
            "flight_id": "FL-123",
            "price": 450,
            "status": "confirmed",
        })

    return (
        "Your flight is booked! NYC to London on July 20. "
        "The total cost was $299. Booking reference: BK-456."
    )


# --- Failing trace: tool returned failure but agent claims success ---
@mlflow.trace(name="travel_agent", span_type=SpanType.AGENT)
def hallucination_fabricated_failed_tool(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, TRAVEL_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "hallucinated_completion", "expected_result": "fail"}
    )

    with mlflow.start_span(name="search_and_book", span_type=SpanType.TOOL) as span:
        span.set_inputs({"from_city": "NYC", "to_city": "London", "date": "2026-07-20"})
        span.set_outputs({"status": "failed", "reason": "no flights available"})

    return (
        "Your flight is booked! NYC to London, July 20. "
        "Booking reference: BK-789. Have a great trip!"
    )


# --- Failing trace: agent claims booking without calling any tool ---
@mlflow.trace(name="travel_agent", span_type=SpanType.AGENT)
def hallucination_fabricated_no_tool(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, TRAVEL_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "hallucinated_completion", "expected_result": "fail"}
    )

    return (
        "Your flight is booked! NYC to London, July 20. "
        "Booking reference: BK-101. Have a great trip!"
    )


# --- Passing trace: agent accurately reflects tool output ---
@mlflow.trace(name="travel_agent", span_type=SpanType.AGENT)
def hallucination_grounded(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, TRAVEL_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "hallucinated_completion", "expected_result": "pass"}
    )

    with mlflow.start_span(name="search_and_book", span_type=SpanType.TOOL) as span:
        span.set_inputs({"from_city": "NYC", "to_city": "London", "date": "2026-07-20"})
        span.set_outputs({
            "booking_id": "BK-456",
            "flight_id": "FL-123",
            "price": 450,
            "status": "confirmed",
        })

    return (
        "Your flight is booked! NYC to London, July 20, flight FL-123. "
        "Total: $450. Booking reference: BK-456."
    )


user_msg = [
    {"role": "user", "content": "Book me a flight from NYC to London on July 20."}
]
hallucination_ungrounded(user_msg)
hallucination_fabricated_failed_tool(user_msg)
hallucination_fabricated_no_tool(user_msg)
hallucination_grounded(user_msg)
mlflow.flush_trace_async_logging()
print("Created 4 traces (3 fail, 1 pass)")

### Load traces

We fetch the Hallucinated Completion traces — three where the agent's response is not grounded in tool evidence, and one where it accurately reflects what tools returned.

In [ ]:
hallucination_traces = mlflow.search_traces(
    experiment_ids=[EXPERIMENT.experiment_id],
    filter_string="tags.failure_mode = 'hallucinated_completion'",
    return_type="list",
)

print(f"Traces found: {len(hallucination_traces)}")
for t in hallucination_traces:
    tags = t.info.tags or {}
    tool_spans = t.search_spans(span_type=SpanType.TOOL)
    print(
        f"  [{tags.get('expected_result', '?')}] "
        f"Tool spans: {len(tool_spans)}  "
        f"Response: {str(t.info.response_preview)[:90]}"
    )
    print()

### Approach: `grounded_in_tools` (custom `@scorer` wrapping MLflow's `is_grounded()`)

MLflow's existing groundedness scorers (`RetrievalGroundedness`) extract context from RETRIEVER spans only — they don't see TOOL span outputs. But the underlying `is_grounded()` judge accepts a `context` parameter directly.

We wrap `is_grounded()` in a custom `@scorer` that:
1. Extracts all TOOL spans from the trace
2. Builds context from each tool's name, inputs, and outputs
3. Passes the context to `is_grounded()` along with the request and response

This teaches a key pattern: reusing an existing MLflow judge for a use case it wasn't designed for by wrapping it in `@scorer` with custom context extraction.

In [ ]:
from mlflow.entities import Feedback, Trace
from mlflow.genai.judges import is_grounded
from mlflow.genai.scorers import scorer


@scorer
def grounded_in_tools(*, trace: Trace) -> Feedback:
    tool_spans = trace.search_spans(span_type=SpanType.TOOL)

    if not tool_spans:
        return Feedback(
            value="no",
            rationale=(
                "No tool calls found in the trace. "
                "The agent's response has no evidence to be grounded in."
            ),
        )

    context = [
        {"content": f"{ts.name}({ts.inputs}) → {ts.outputs}"} for ts in tool_spans
    ]

    root = trace.data.spans[0]
    request = str(root.inputs)
    response = str(root.outputs)

    # name= must match the scorer function name — without it, is_grounded()
    # defaults to "groundedness", creating split metric names with the
    # short-circuit path above (which gets auto-renamed to "grounded_in_tools")
    return is_grounded(
        request=request,
        response=response,
        context=context,
        name="grounded_in_tools",
        model="openai:/gpt-4o",
    )


with mlflow.start_run(run_name="hallucinated-completion-grounded-in-tools") as run:
    results = mlflow.genai.evaluate(
        data=hallucination_traces,
        scorers=[grounded_in_tools],
    )

print_eval_results(results, "grounded_in_tools", EXPERIMENT.experiment_id)

### Interpreting the results

- **Ungrounded response** (wrong price) → `no` — the agent said $299 but the tool returned $450. The judge detects the contradiction.
- **Fabricated action — tool failed** → `no` — the tool returned `{"status": "failed"}` but the agent claimed the booking succeeded. The judge sees the tool evidence contradicts the claim.
- **Fabricated action — no tool called** → `no` — no tool spans exist in the trace, so the scorer short-circuits: the agent's claims have zero evidence to be grounded in.
- **Grounded response** → `yes` — the agent's response accurately reflects the booking ID, flight ID, and price from the tool output.

**Why wrap `is_grounded()` instead of using `RetrievalGroundedness` directly?** `RetrievalGroundedness` extracts context from RETRIEVER spans only. For agents that use TOOL spans (not retrievers), it finds no context and can't evaluate groundedness. By wrapping the underlying `is_grounded()` judge in `@scorer`, we extract tool outputs ourselves and pass them as context — extending the judge's capabilities beyond its default RAG-only behavior.

**Note on models:** This scorer uses `gpt-4o` (explicitly specified; the MLflow default is `gpt-4.1-mini`). You can change the `model=` parameter in the `is_grounded()` call to use any provider MLflow supports.

**Note:** The `is_grounded()` call uses an LLM judge, so results may vary slightly between runs. The verdicts above are the expected outcomes for these traces, but LLM judges are non-deterministic — borderline cases may occasionally be judged differently. The "no tool called" case is deterministic (short-circuit, no LLM involved).

For full details on how the scorer works and the groundedness prompt, see [hallucinated_completion.md](hallucinated_completion.md).